In [1]:
import json

with open("my_chunks_sections.json", "r", encoding="utf-8") as f:
    my_chunks_sections = json.load(f)

len(my_chunks_sections)
my_chunks_sections[0]

{'filename': 'binance-spot-api-docs-master/CHANGELOG_CN.md',
 'section': '## REST API\n\n- `POST /api/v3/userDataStream`\n- `PUT /api/v3/userDataStream`\n- `DELETE /api/v3/userDataStream`'}

In [2]:
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

In [3]:
def normalize_binance_chunks(chunks):
    normalized = []

    for i, doc in enumerate(chunks):
        item = dict(doc)

        chunk_text = (
            item.get("chunk")
            or item.get("content")
            or item.get("text")
            or ""
        )

        title = item.get("title", "") or item.get("heading", "") or ""
        filename = (
            item.get("filename", "")
            or item.get("file_path", "")
            or item.get("path", "")
            or item.get("source", "")
            or ""
        )
        section = item.get("section", "") or item.get("header", "") or ""
        description = item.get("description", "") or ""

        normalized.append({
            "id": item.get("id", str(i)),
            "chunk": chunk_text,
            "title": title,
            "filename": filename,
            "section": section,
            "description": description,
            "url": item.get("url", ""),
            "raw": item
        })

    return normalized

In [4]:
binance_docs = []

for i, d in enumerate(my_chunks_sections):
    binance_docs.append({
        "id": str(i),
        "chunk": d.get("chunk", "") if isinstance(d, dict) else "",
        "title": d.get("title", "") if isinstance(d, dict) else "",
        "description": d.get("description", "") if isinstance(d, dict) else "",
        "filename": d.get("filename", "") if isinstance(d, dict) else "",
        "section": d.get("section", "") if isinstance(d, dict) else ""
    })

len(binance_docs)

367

In [5]:
binance_docs[0]

{'id': '0',
 'chunk': '',
 'title': '',
 'description': '',
 'filename': 'binance-spot-api-docs-master/CHANGELOG_CN.md',
 'section': '## REST API\n\n- `POST /api/v3/userDataStream`\n- `PUT /api/v3/userDataStream`\n- `DELETE /api/v3/userDataStream`'}

In [6]:
text_index = Index(
    text_fields=["chunk", "title", "description", "filename", "section"],
    keyword_fields=[]
)

text_index.fit(binance_docs)

In [7]:
query = "How do I sign a Binance Spot API request?"
text_results = text_index.search(query, num_results=5)
text_results

[{'id': '170',
  'chunk': '',
  'title': '',
  'description': '',
  'filename': 'binance-spot-api-docs-master/fix-api.md',
  'section': '## General API Information\n\n* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.\n* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)\n  * This does not always mean that the request failed in the Matching Engine.\n  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.\n* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.\

In [8]:
def preview_results(results, max_chars=300):
    for i, r in enumerate(results, 1):
        print(f"\nResult #{i}")
        print("title:      ", r.get("title", ""))
        print("section:    ", r.get("section", ""))
        print("filename:   ", r.get("filename", ""))
        print("chunk:      ", r.get("chunk", "")[:max_chars])
        print("-" * 80)

In [9]:
preview_results(text_results)


Result #1
title:       
section:     ## General API Information

* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.
* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)
  * This does not always mean that the request failed in the Matching Engine.
  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* To ensure uninterrupted connectivity, please make sure that your client sends **SNI (Server Name Indica

In [10]:
embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
binance_embeddings = []

for d in tqdm(binance_docs):
    v = embedding_model.encode(d["chunk"])
    binance_embeddings.append(v)

binance_embeddings = np.array(binance_embeddings)
binance_embeddings.shape

  0%|          | 0/367 [00:00<?, ?it/s]

(367, 768)

In [12]:
vector_index = VectorSearch()
vector_index.fit(binance_embeddings, binance_docs)

In [13]:
query = "How can I authenticate a private Binance API endpoint?"
q = embedding_model.encode(query)

vector_results = vector_index.search(q, num_results=5)
preview_results(vector_results)


Result #1
title:       
section:     ## 如何正确在本地维护一个order book副本

1. 打开与 `wss://stream.binance.com:9443/ws/bnbbtc@depth` 的 WebSocket 连接。
2. 开始缓存收到的event。请记录您收到的第一个event的 `U`值。
3. 访问 `https://api.binance.com/api/v3/depth?symbol=BNBBTC&limit=5000` 获取深度快照。
4. 如果快照中的 `lastUpdateId` 小于等于步骤 2 中的 `U` 值，请返回步骤 3。
5. 在收到的event中，丢弃快照中 `u` <= `lastUpdateId` 的所有event。现在第一个event的 `lastUpdateId` 应该在 `[U;u]` 范围以内。
6. 将本地order book设置为快照。它的更新ID 为 `lastUpdateId`。
7. 更新所有缓存的event，以及后续的所有event。

要将一个event应用于您的本地order book，请遵循以下更新过程：
1. 判断是否需要处理event：
    * 如果event的最后一次更新ID（`u`）小于本地order book的更新ID，忽略该event。
    * 如果event的首次更新ID（`U`）大于本地order book的更新ID加1，说明你错过了一些events。<br>请丢弃您的本地order book并从头开始重新同步。
    * 通常，下一event的`U`等于上一event的`u + 1`。
1. 对买价（`b`）和卖价（`a`）中的每个价位，设置order book中的新数量：
    * 如果该价位在order book中不存在，则插入该价位及其数量。
    * 如果数量为零，则从order book中删除此价位。
1. 将order book的更新ID设置为已处理event的最后一次更新ID（`u`）。

> [!NOTE]
> 由于从 API 检索的深度快照对价位的数量有限制（每侧最多 5000 个），因此除非它们发生变化，否则您将无法了解初始快照之外的价位数量。<br>
> 因此，在使用这些级别的信息时要小心，因为它们

In [14]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vector_index.search(q, num_results=num_results)

In [15]:
def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)

    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result["id"] not in seen_ids:
            seen_ids.add(result["id"])
            combined_results.append(result)

    return combined_results[:num_results]

In [16]:
query = "What does recvWindow mean in Binance API requests?"
hybrid_results = hybrid_search(query, num_results=5)
preview_results(hybrid_results)


Result #1
title:       
section:     ## General API Information

* The following base endpoints are available. Please use whichever works best for your setup:
  * **https://api.binance.com**
  * **https://api-gcp.binance.com**
  * **https://api1.binance.com**
  * **https://api2.binance.com**
  * **https://api3.binance.com**
  * **https://api4.binance.com**
* The last 4 endpoints in the point above (`api1`-`api4`) should give better performance but have less stability.
* Responses are in JSON by default. To receive responses in SBE, refer to the [SBE FAQ](./faqs/sbe_faq.md) page.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* Some endpoints may return asset and/or symbol names containing non-ASCII characters encoded in UTF-8 even if the request did not contain non-ASCII characters.
* Data is returned in **chronological order**, unless noted otherwise.
  * Without `startTime` or `endTime`, 

In [17]:
test_queries = [
    "How do I sign a Binance Spot API request?",
    "What does recvWindow mean?",
    "Why am I getting timestamp error?",
    "How do I place a market order?",
    "How can I authenticate a private endpoint?",
    "What are the rate limits?",
    "How do I get exchange info?"
]

for query in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)
    results = hybrid_search(query, num_results=3)
    preview_results(results, max_chars=220)


QUERY: How do I sign a Binance Spot API request?

Result #1
title:       
section:     ## General API Information

* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.
* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)
  * This does not always mean that the request failed in the Matching Engine.
  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* To ensure uninterrupted connectivity, please make sur

In [18]:
len(my_chunks_sections)

367

In [19]:
preview_results(text_search("What does recvWindow mean?"))


Result #1
title:       
section:     ## In the API response, there's a field called `workingFloor`. What does that field mean?

This is a term used to determine where the order's last activity occurred (filling, expiring, or being placed as new, etc.).

If the `workingFloor` is `SOR`, this means your order interacted with other eligible order books in the SOR configuration.

If the `workingFloor` is `EXCHANGE`, this means your order interacted on the order book that you sent that order to.
filename:    binance-spot-api-docs-master/faqs/sor_faq.md
chunk:       
--------------------------------------------------------------------------------

Result #2
title:       
section:     ## What does the Price Range Execution Rule do?

This rule ensures that trades may only be executed at prices within and equal to a price range around a reference price.
filename:    binance-spot-api-docs-master/faqs/price_range_execution_rules.md
chunk:       
---------------------------------------------------

In [20]:
preview_results(vector_search("How do I authenticate a private endpoint?"))


Result #1
title:       
section:     ## 如何正确在本地维护一个order book副本

1. 打开与 `wss://stream.binance.com:9443/ws/bnbbtc@depth` 的 WebSocket 连接。
2. 开始缓存收到的event。请记录您收到的第一个event的 `U`值。
3. 访问 `https://api.binance.com/api/v3/depth?symbol=BNBBTC&limit=5000` 获取深度快照。
4. 如果快照中的 `lastUpdateId` 小于等于步骤 2 中的 `U` 值，请返回步骤 3。
5. 在收到的event中，丢弃快照中 `u` <= `lastUpdateId` 的所有event。现在第一个event的 `lastUpdateId` 应该在 `[U;u]` 范围以内。
6. 将本地order book设置为快照。它的更新ID 为 `lastUpdateId`。
7. 更新所有缓存的event，以及后续的所有event。

要将一个event应用于您的本地order book，请遵循以下更新过程：
1. 判断是否需要处理event：
    * 如果event的最后一次更新ID（`u`）小于本地order book的更新ID，忽略该event。
    * 如果event的首次更新ID（`U`）大于本地order book的更新ID加1，说明你错过了一些events。<br>请丢弃您的本地order book并从头开始重新同步。
    * 通常，下一event的`U`等于上一event的`u + 1`。
1. 对买价（`b`）和卖价（`a`）中的每个价位，设置order book中的新数量：
    * 如果该价位在order book中不存在，则插入该价位及其数量。
    * 如果数量为零，则从order book中删除此价位。
1. 将order book的更新ID设置为已处理event的最后一次更新ID（`u`）。

> [!NOTE]
> 由于从 API 检索的深度快照对价位的数量有限制（每侧最多 5000 个），因此除非它们发生变化，否则您将无法了解初始快照之外的价位数量。<br>
> 因此，在使用这些级别的信息时要小心，因为它们

In [21]:
preview_results(hybrid_search("Why am I getting timestamp error?"))


Result #1
title:       
section:     ## Error Codes

* Any endpoint can return an ERROR

Sample Payload below:
```javascript
{
    "code": -1121,
    "msg": "Invalid symbol."
}
```
* Specific error codes and messages are defined in [Errors Codes](./errors.md).
filename:    binance-spot-api-docs-master/rest-api.md
chunk:       
--------------------------------------------------------------------------------

Result #2
title:       
section:     ## Error Codes

* Any endpoint can return an ERROR

Sample Payload below:
```javascript
{
    "code": -1121,
    "msg": "Invalid symbol."
}
```
* Specific error codes and messages are defined in [Errors Codes](./errors.md).
filename:    binance-spot-api-docs-master/testnet/rest-api.md
chunk:       
--------------------------------------------------------------------------------

Result #3
title:       
section:     ## 追踪止盈止损下单示例

假设当前最新价是40,000.

下一个追踪止损(`STOP_LOSS_LIMIT`)的买单, 设置价格为42,000, 追踪价差（`trailingDelta`)为5%.

```bash
# 不设置止损价(stop price)


In [ ]:
#Text search works well for exact Binance terms like recvWindow, timestamp, and endpoint names.
#TVector search helps when the question is paraphrased.
#THybrid search is the safest overall.
#TFor this project, I would start with text search and keep hybrid as the stronger production option.